In [1]:
#### Core script to defines and execute rf runs using functions defined in the following scripts

# config          # configures the model run with input data and defaults (where not directly input)
# spatial_cv      # set up the spatial CV blocks based on fold CSV
# train           # trains model and predicts on test data
# evaluate        # calculates metrics for each fold and overall 
# importance      # calculates feature importance + permutation importance for each fold and overall 

from pathlib import Path
import json
import dataclasses
import pandas as pd

from step_a_config import RunConfig
from step_b_spatial_cv import make_spatial_folds
from step_c_train import train_model
from step_d_evaluate import evaluate
from step_e_importance import get_feature_importance
from step_f_evaluate_original import run_intensity_evaluation

In [2]:
##### Set-up (doesn't change)

# Get the current working directory
cd = Path.cwd().parent.parent 

# set directories
DATA_DIR = Path(f"{cd}/Data/Clean/Training_data/")
RESULTS_DIR = Path(f"{cd}/Results/RF_models/")
FOLD_DIR = Path(f"{cd}/Data/Fold_assignments/") 

In [3]:
#### DEFINE RUN (MANUAL)

version = 'capital' # capital or labor
unit = 'tonne' # USD or tonne
variable_def = 'rtv' # abs or rtv
type_of_model = 'rf'  # rf, qrf, xgb, or lgbm
sample_weighting_option = 'sqrt_inverse'   # "none", "inverse", or "sqrt_inverse"

name_of_run =  f'{version}_{unit}_{variable_def}_{type_of_model}_{sample_weighting_option}'

In [4]:
##### Configuration (some manual changes)

# set target variable 
numerator = "USD" if version == "capital" else "jobs"
precedent = "rtv_" if variable_def == "rtv" else ""
target_var = f'{precedent}log_{version}_intensity_{numerator}_per_million_{unit}' 
 
# config
model_data_type = "absolute" if variable_def == "abs" else "relative"

fold = FOLD_DIR / f"{version}_folds.csv"
model_data = f"{version}_{model_data_type}_final_thinned.csv"

RUNS = [
    RunConfig(
        run_name         = name_of_run,
        target           = target_var,   
        dataset          = model_data,
        fold_assignments = fold,
        model_type       = type_of_model,
        id_cols          = ["PROJ_ID", "country_ID"],
        weighting        = sample_weighting_option, 
        version          = version,
        unit             = unit,
        variable_def     = variable_def
    ),
]

# define which columns are feature columns 
feature_cols = ['rtv_log_SOC', 'rtv_log_average_travel_time_city',
       'rtv_log_average_travel_time_port',
       'rtv_log_growing_season_length_days', 'rtv_log_GDP_pc', 'rtv_log_slope',
       'rtv_log_crop_intensity',
       'rtv_log_USD_production_per_million_HA',
       'rtv_log_tonnes_production_per_million_HA',
       'rtv_log_pop_density_people_per_100_km2',
       'rtv_log_buffalo_density_per_100_km2',
       'rtv_log_chicken_density_per_100_km2',
       'rtv_log_cattle_density_per_100_km2',
       'rtv_log_goats_density_per_100_km2', 'rtv_log_pigs_density_per_100_km2',
       'rtv_log_sheep_density_per_100_km2',
       'rtv_log_livestock_density_LU_per_100_km2',
       'rtv_log_female_share_base_100',
       'rtv_log_cereals_share_base_100_production_USD',
       'rtv_log_fibres_share_base_100_production_USD',
       'rtv_log_fruits_share_base_100_production_USD',
       'rtv_log_oilcrops_share_base_100_production_USD',
       'rtv_log_pulses_share_base_100_production_USD',
       'rtv_log_roots_tubers_share_base_100_production_USD',
       'rtv_log_rest_of_crops_share_base_100_production_USD',
       'rtv_log_sugar_crops_share_base_100_production_USD',
       'rtv_log_vegetables_share_base_100_production_USD',
       'rtv_log_rubber_share_base_100_production_USD',
       'rtv_log_ruminants_share_base_100_production_USD',
       'rtv_log_monogastrics_share_base_100_production_USD',
       'rtv_log_poultry_share_base_100_production_USD',
       'rtv_log_child_dependency_ratio_per_hundred',
       'rtv_probability_survival_land_use_objective',
       'rtv_probability_economic_land_use_objective']

In [5]:
##### Define function to save results into defined directory
def save_results(results, config, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True) # creates folder if it doesn't already exist 

    # save config details for run 
    # convert any Path objects to strings so json can handle them
    config_dict = dataclasses.asdict(config)
    config_dict = {
        k: str(v) if isinstance(v, Path) else v
        for k, v in config_dict.items()
    }
    (out_dir / "config.json").write_text(
        json.dumps(config_dict, indent=2)
    )

    # save predictions across all folds
    results["predictions"].to_parquet(out_dir / "predictions.parquet", index=False)

    # save the best parameters for each fold
    params_df = pd.DataFrame([
        {"fold": r["fold"], **r["best_params"]}
        for r in results["fold_results"]
    ])
    params_df.to_csv(out_dir / "best_params.csv", index=False)

    # save metrics and importance scores
    results["metrics"].to_csv(out_dir / "metrics.csv", index=False)
    results["country_metrics"].to_csv(out_dir / "country_metrics.csv", index=False)
    results["importance"]["combined"].to_csv(out_dir / "importance.csv", index=False)
    results["importance"]["top_n_stability"].to_csv(out_dir / "importance_top_n_stability.csv", index=False)
    
    # add intensity metrics
    if "intensity_metrics" in results:
        results["intensity_metrics"].to_csv(out_dir / "intensity_metrics.csv", index=False)

    print(f"  saved to {out_dir}")

In [6]:
##### RUN   

# define the function to run all scripts in order 
def run(config):
    print(f"\n{'═'*60}")
    print(f"  run: {config.run_name}")
    print(f"  target: {config.target}  |  model: {config.model_type}")
    print(f"{'═'*60}")

    # load model data
    df = pd.read_csv(DATA_DIR / config.dataset)

    # run spatial folds functions 
    print("\n── Spatial folds ────────────────────────────────────────")
    folds = make_spatial_folds(df, config)

    # run training and prediction functions
    print("\n── Training ─────────────────────────────────────────────")
    results = train_model(df, feature_cols, folds, config)

    # run evaluation functions
    results["metrics"], results["country_metrics"] = evaluate(results, config)

    if config.version and config.unit:
        print("\n── Absolute intensity evaluation (test only) ────────")
        results["intensity_metrics"], _ = run_intensity_evaluation(
            results, config,
            country_intensity_path=Path(f"{cd}/Data/Clean/Intensities/country_intensities.csv"),
            intensity_dir=Path(f"{cd}/Data/Clean/Intensities/"),
    )

    # run importance functions
    print("\n── Feature importance ───────────────────────────────────")
    results["importance"] = get_feature_importance(results, df, config)

    # run save function
    print("\n── Saving ───────────────────────────────────────────────")
    out_dir = RESULTS_DIR / config.run_name
    save_results(results, config, out_dir)


# Run run function for each model configuration defined above
for config in RUNS:
    run(config)


════════════════════════════════════════════════════════════
  run: capital_tonne_rtv_rf_sqrt_inverse
  target: rtv_log_capital_intensity_USD_per_million_tonne  |  model: rf
════════════════════════════════════════════════════════════

── Spatial folds ────────────────────────────────────────
  fold_1: 25 train countries (1,941 rows) | 1 test countries (1,090 rows)
  fold_2: 25 train countries (2,759 rows) | 1 test countries (272 rows)
  fold_3: 25 train countries (1,972 rows) | 1 test countries (1,059 rows)
  fold_4: 25 train countries (2,766 rows) | 1 test countries (265 rows)
  fold_5: 22 train countries (345 rows) | 4 test countries (2,686 rows)
  fold_6: 5 train countries (2,686 rows) | 21 test countries (345 rows)
  fold_7: 11 train countries (2,913 rows) | 15 test countries (118 rows)
  fold_8: 20 train countries (2,804 rows) | 6 test countries (227 rows)

── Training ─────────────────────────────────────────────

── fold_1 ──────────────────────────────
  test countries: ['BRA